# **Data cleaning**

We load the merged raw *Global Wellbeing, Sustainability and Resource Use Dataset*. 

We screen the data and look for missing values, specifically trying to identify countries with a substantial amount of missing values in the key variables:

| Key variables | Description | Source(s) |
|---|---|---|
| `happiness_index` | Measure of Subjective Wellbeing | World Happiness Report (https://www.kaggle.com/datasets/simonaasm/world-happiness-index-by-reports-2013-2023) |
| `gini_index` | Measure of income inequality | World Bank GINI Index (https://data.worldbank.org/indicator/SI.POV.GINI) |
| `consumption_co2_per_capita` | Consumption-based CO₂ emissions per capita | Our World in Data CO₂ (https://github.com/owid/co2-data) |
| `co2_per_capita` | Production-based CO₂ emissions per capita | Our World in Data CO₂ Data (https://github.com/owid/co2-data) |
| `renewables_consumption` | Share of primary energy consumption from renewable sources | Our World in Data Energy (https://github.com/owid/energy-data) |
| `energy_per_capita` | Primary energy consumption per capita | Our World in Data Energy (https://github.com/owid/energy-data) (See below) |
| `material_footprint_per_capita` | Per capita material consiumption indicator | UN Human Development Reports (https://www.kaggle.com/datasets/iamsouravbanerjee/material-footprint-per-capita-by-country) |

This will help us identify countries that may be more reasonable to drop than keep and handle missing values of. Relative to our variables of interest they contain more noise than actual information.

In [1]:
import pandas as pd

In [2]:
# Make imports from the scr/ directory work.
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

In [3]:
from scr.io import load_csv
from scr.config import RAW_PATH

df_raw = load_csv(RAW_PATH)
display(df_raw.head())

df_raw.info()

,country,iso_code,year,co2_per_capita,consumption_co2_per_capita,energy_per_capita_x,temperature_change_from_co2,share_global_co2,land_use_change_co2_per_capita,population,...,renewables_consumption,happiness_index,happiness_index_rank,continent,hemisphere,human_development_groups,hdi_rank_2021,undp_developing_regions,material_footprint_per_capita,gini_index
0,Aruba,ABW,2013,8.395,NaN,47742.637,0.0,0.002,NaN,102570.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aruba,ABW,2014,8.435,NaN,47990.926,0.0,0.002,NaN,103381.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aruba,ABW,2015,8.615,NaN,48905.531,0.0,0.003,NaN,104200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aruba,ABW,2016,8.411,NaN,47619.418,0.0,0.002,NaN,104989.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aruba,ABW,2017,8.420,NaN,49061.195,0.0,0.002,NaN,105737.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 1962 entries, 0 to 1961
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         1962 non-null   str    
 1   iso_code                        1962 non-null   str    
 2   year                            1962 non-null   int64  
 3   co2_per_capita                  1917 non-null   float64
 4   consumption_co2_per_capita      1079 non-null   float64
 5   energy_per_capita_x             1836 non-null   float64
 6   temperature_change_from_co2     1935 non-null   float64
 7   share_global_co2                1935 non-null   float64
 8   land_use_change_co2_per_capita  1746 non-null   float64
 9   population                      1845 non-null   float64
 10  gdp                             1476 non-null   float64
 11  energy_per_capita_y             1836 non-null   float64
 12  renewables_consumption          711 non-null 

We notice we have two energy_per_capita variables, _x and _y. These come from two different datasets preceding the merge into a single dataset.
1. We consider both to establish whether they are different in any way.
2. If different, we must establish a criterion for keeping one over the other - first consider if any of the two has more missing values.

In [4]:
print(df_raw['energy_per_capita_x'].isna().sum())
print(df_raw['energy_per_capita_y'].isna().sum())

# We now identify which countries and years have missing data in each variable

print(df_raw[df_raw['energy_per_capita_x'].isna()]['country'].unique())
print(df_raw[df_raw['energy_per_capita_y'].isna()]['country'].unique())

# Final check that the non-missing values are the same in both variables, so we can safely drop one of them.
# We do this by computing their differences in every row, storing the results into a set, and then checking if the set contains only zero.

print(set(df_raw[df_raw['energy_per_capita_x'].notna()]['energy_per_capita_x'] - df_raw[df_raw['energy_per_capita_y'].notna()]['energy_per_capita_y']))

# There seem to be some differences. 
# We must find which values are different between the two variables 
# and print them as a list indexed by country and year 
# in order to understand where the differences are.

# Find rows where both values exist AND they're different
mask = (df_raw['energy_per_capita_x'].notna() & 
        df_raw['energy_per_capita_y'].notna() & 
        (df_raw['energy_per_capita_x'] != df_raw['energy_per_capita_y']))

print(df_raw[mask][['iso_code', 'year', 'energy_per_capita_x', 'energy_per_capita_y']])


126
126
<StringArray>
[                       'Anguilla',                         'Andorra',
                      'Antarctica', 'Bonaire Sint Eustatius and Saba',
                         'Curacao',                'Christmas Island',
                   'Liechtenstein',                          'Monaco',
                'Marshall Islands',                           'Palau',
                      'San Marino',       'Sint Maarten (Dutch part)',
                         'Vatican',               'Wallis and Futuna']
Length: 14, dtype: str
<StringArray>
[                       'Anguilla',                         'Andorra',
                      'Antarctica', 'Bonaire Sint Eustatius and Saba',
                         'Curacao',                'Christmas Island',
                   'Liechtenstein',                          'Monaco',
                'Marshall Islands',                           'Palau',
                      'San Marino',       'Sint Maarten (Dutch part)',
                  

So all differences come from a single country, Togo (iso_code: TGO), where `energy_per_capita_y > energy_per_capita_x`

Looking at the source of the original datasets, the energy dataset (y) has been updated more recently than the CO2 dataset (x) - 3 weeks vs 5 months. This aligns with the observation prior to merging, that for the same countries, the energy dataset had greater richness in the population and gdp variables - which where consequently kept to maximise the availability of real information.

Following along these lines, we make the choice to keep the y-variable, from energy data, rather than taking an average between the two.

In [5]:
# We must now rename the energy_per_capita_y variable to energy_per_capita, and drop the _x version
df_raw = df_raw.rename(columns={'energy_per_capita_y': 'energy_per_capita'})
df_raw = df_raw.drop(columns=['energy_per_capita_x'])

In [6]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
    ]

missing_by_year = (
    df_raw[key_vars]
    .isna()
    .groupby(df_raw['year'])
    .sum()
    )

display(missing_by_year)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
year,,,,,,,
2013,68,142,61,98,5,14,139
2014,218,135,61,98,5,14,139
2015,67,132,61,98,5,14,139
2016,68,136,61,99,5,14,139
2017,68,141,61,98,5,14,139
2018,67,126,61,98,5,14,139
2019,68,141,61,98,5,14,139
2020,71,149,61,98,5,14,139
2021,74,138,61,98,5,14,139


We select key variables and identify the number of missing values per iso_code/country (where there are any) aggregated over the years.

To aid with this process in a managable manner given the large number of countries, we build a function to:
1. Compute missing values by country-year.
2. Identify country-years with more than N missing variables.
3. Extract the affected countries.
4. Return the country-level missingness summary for only those countries.

In [7]:
from scr.utils import countries_with_missing_vars

key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
    ]

subset_missing_6 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=6
)

display(subset_missing_6)

# We can see that the only countries with missing data are Antarctica, Christmas Island, Monaco, San Marino, and Vatican
# We can drop these countries from our dataset by first finding their iso_code from df_raw and then dropping them from the dataframe

iso_codes_to_drop = df_raw[df_raw['country'].isin(subset_missing_6.index)]['iso_code'].unique()
df_raw = df_raw[~df_raw['iso_code'].isin(iso_codes_to_drop)]

# Confirm that the countries with missing data have been dropped
new_subset_missing_6 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=6
)

display(new_subset_missing_6)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Antarctica,9,9,9,9,9,9,9
Christmas Island,9,9,9,9,9,9,9
Monaco,9,9,9,9,9,9,9
San Marino,9,9,9,9,9,9,9
Vatican,9,9,9,9,9,9,9


,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


We now look for countries with missing values for 6 of the 7 key variables.

We see that 9 countries, out of all the key variables, only have observations for all years for co2_per_capita. This is not enough to justify keeping them in the dataset.

In [8]:
subset_missing_5 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=5
)

display(subset_missing_5)

iso_codes_to_drop = df_raw[df_raw['country'].isin(subset_missing_5.index)]['iso_code'].unique()
df_raw = df_raw[~df_raw['iso_code'].isin(iso_codes_to_drop)]

new_subset_missing_5 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=5
)

display(new_subset_missing_5)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Andorra,9,9,9,9,0,9,9
Anguilla,9,9,9,9,0,9,9
Bonaire Sint Eustatius and Saba,9,9,9,9,0,9,9
Curacao,9,9,9,9,0,9,9
Liechtenstein,9,9,9,9,0,9,9
Marshall Islands,9,8,9,9,0,9,9
Palau,9,9,9,9,0,9,9
Sint Maarten (Dutch part),9,9,9,9,0,9,9
Wallis and Futuna,9,9,9,9,0,9,9


,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


We now look for countries with missing values for 5 of the 7 key variables using the function we built again.

- 42 of the remaining countries only have values for co2_per_capita and energy_per_capita, for all years consistently in all cases.
- Some of the countries have some sparse readings for happiness and gini indices, which means that at best they have valid entries for 3 of the 7 key variables, and only for some years. This is less than 50%, so we risk introducing more noise than information into the data.
- The decision is made to drop all the identified countries on the basis of too many missing values with a consistent pattern that is not recoverable in a meaningful manner.


In [9]:
subset_missing_4 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=4
)

display(subset_missing_4)

iso_codes_to_drop = df_raw[df_raw['country'].isin(subset_missing_4.index)]['iso_code'].unique()
df_raw = df_raw[~df_raw['iso_code'].isin(iso_codes_to_drop)]

new_subset_missing_4 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=4
)

display(new_subset_missing_4)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Antigua and Barbuda,9,9,9,9,0,0,9
Aruba,9,9,9,9,0,0,9
Barbados,9,8,9,9,0,0,9
Bermuda,9,9,9,9,0,0,9
British Virgin Islands,9,9,9,9,0,0,9
Cape Verde,9,8,9,9,0,0,9
Comoros,3,7,9,9,0,0,9
Cook Islands,9,9,9,9,0,0,9
Dominica,9,9,9,9,0,0,9


,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


In [10]:
print(f"There are {df_raw['country'].nunique()} countries in the dataset.")

There are 162 countries in the dataset.


We check for countries missing 4/7 key vars.

Here the search becomes more challenging, the main pattern that can be observed is:
- All identified countries consistently have all values for energy_per_capita and co2_per_capita, and in most cases material_footrpint_per_capita.
- Many of them only have one missing year in the happiness_index variable. But this was the case for all countries:
    - To aid with navigating this more nuanced scenario, we first impute the missing values in year 2014 in the happiness index variable.

In [11]:
subset_missing_3 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=3
)

display(subset_missing_3)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Afghanistan,1,9,0,9,0,0,9
Angola,4,8,0,9,0,0,9
Bahamas,9,9,0,9,0,0,9
Belize,6,8,0,9,0,0,9
Bhutan,4,8,0,9,0,0,9
Bosnia and Herzegovina,1,7,0,9,0,0,9
Burundi,1,7,0,9,0,0,9
Central African Republic,3,8,0,9,0,0,9
Chad,1,8,0,9,0,0,9


In [12]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
    ]

missing_by_year = (
    df_raw[key_vars]
    .isna()
    .groupby(df_raw['year'])
    .sum()
    )

display(missing_by_year)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
year,,,,,,,
2013,16,90,5,42,0,0,83
2014,162,81,5,42,0,0,83
2015,15,79,5,42,0,0,83
2016,15,84,5,43,0,0,83
2017,14,87,5,42,0,0,83
2018,13,73,5,42,0,0,83
2019,15,90,5,42,0,0,83
2020,19,94,5,42,0,0,83
2021,22,84,5,42,0,0,83


We note that all 2014 values for happiness index are missing. We interpolate these by country linearly between the previous and next years to 2014.

In [13]:
# Check how many 2014 missing values were filled
print(f"2014 missing before: {df_raw[(df_raw['year'] == 2014) & (df_raw['happiness_index'].isnull())].shape[0]}")

# Create a Series with interpolated values for all rows
interpolated = df_raw.groupby('iso_code')['happiness_index'].transform(lambda x: x.interpolate(method='linear'))

# Only fill 2014 missing values
mask_2014 = (df_raw['year'] == 2014) & (df_raw['happiness_index'].isnull())
df_raw.loc[mask_2014, 'happiness_index'] = interpolated[mask_2014]


print(f"2014 missing after: {df_raw[(df_raw['year'] == 2014) & (df_raw['happiness_index'].isnull())].shape[0]}")

2014 missing before: 162
2014 missing after: 16


In [14]:
subset_missing_3 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=3
)

display(subset_missing_3)


,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Angola,3,8,0,9,0,0,9
Bahamas,9,9,0,9,0,0,9
Belize,6,8,0,9,0,0,9
Bhutan,4,8,0,9,0,0,9
Central African Republic,2,8,0,9,0,0,9
Congo,9,9,0,9,0,0,9
Cuba,9,9,0,9,0,0,9
Democratic Republic of Congo,9,8,0,9,0,0,9
Djibouti,6,7,0,9,0,0,9


This interpolation nearly halved the number of countries missing 4/7 key vars.

Since these countries are missing consumption_co2_per_capita and most entries for happiness_index and gini_index, we do not have enough data to look at the desired features for these countries.

In [15]:
iso_codes_to_drop = df_raw[df_raw['country'].isin(subset_missing_3.index)]['iso_code'].unique()
df_raw = df_raw[~df_raw['iso_code'].isin(iso_codes_to_drop)]

new_subset_missing_3 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=3
)
display(new_subset_missing_3)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


In [16]:
print(f"There are {df_raw['country'].nunique()} countries in the dataset.")

There are 144 countries in the dataset.


In [17]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "renewables_consumption"
    ]

subset_missing_2 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=2
)

display(subset_missing_2)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,renewables_consumption
country,,,,,
Afghanistan,0,9,0,9,9
Bosnia and Herzegovina,0,7,0,9,9
Brunei,9,9,0,0,9
Burundi,0,7,0,9,9
Chad,0,8,0,9,9
Cote d'Ivoire,9,6,0,0,9
Gabon,0,8,0,9,9
Haiti,0,9,0,9,9
Laos,1,8,0,0,9


Material consumption and consumption co2 are central to the analysis of interest.

We can safaly drop countries that are still at this point missing vlues for all years of both these columns.

We do the same criterion but look for countries that are missing more than 3 years in both happiness index and gini_index - since these are the key social indicators.

In [18]:
# Step 1: Find rows in subset_missing_2 where both variables are missing
key_missing = subset_missing_2[
    (subset_missing_2['material_footprint_per_capita']==9) & 
    (subset_missing_2['consumption_co2_per_capita']==9)
    |
    (subset_missing_2['happiness_index']>3) &
    (subset_missing_2['gini_index']>3)
]

# Step 2: Get unique iso_code from these rows
countries_to_drop = df_raw[df_raw['country'].isin(key_missing.index)]['iso_code'].unique()

# Step 3: Display countries to be dropped
print(f"Countries to drop: {countries_to_drop}")
print(f"Number of countries: {len(countries_to_drop)}")

# Step 4: Remove these countries from df_raw
df_raw = df_raw[~df_raw['iso_code'].isin(countries_to_drop)]

# Step 5: Verify removal
print(f"Remaining unique countries: {df_raw['iso_code'].nunique()}")

Countries to drop: <StringArray>
['BRN', 'CIV', 'MNE']
Length: 3, dtype: str
Number of countries: 3
Remaining unique countries: 141


In [19]:
subset_missing_2 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=2
)
display(subset_missing_2)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,renewables_consumption
country,,,,,
Afghanistan,0,9,0,9,9
Bosnia and Herzegovina,0,7,0,9,9
Burundi,0,7,0,9,9
Chad,0,8,0,9,9
Gabon,0,8,0,9,9
Haiti,0,9,0,9,9
Laos,1,8,0,0,9
Lebanon,0,9,0,9,9
Liberia,0,7,0,9,9


All countries missing all gini_index entries are also missing either all consumption co2 or material footprint data. We can safely drop these as they are missing too many key components.

In [20]:
# Step 1: Find rows in subset_missing_2 where all gini_index entries are missing
gini_missing = subset_missing_2[
    (subset_missing_2['gini_index']==9)
    & ((subset_missing_2['consumption_co2_per_capita']==9)
       | (subset_missing_2['material_footprint_per_capita']==9))
]

# Step 2: Get unique iso_code from these rows
countries_to_drop = df_raw[df_raw['country'].isin(gini_missing.index)]['iso_code'].unique()

# Step 3: Display countries to be dropped
print(f"Countries to drop: {countries_to_drop}")
print(f"Number of countries: {len(countries_to_drop)}")

# Step 4: Remove these countries from df_raw
df_raw = df_raw[~df_raw['iso_code'].isin(countries_to_drop)]

# Step 5: Verify removal
print(f"Remaining unique countries: {df_raw['iso_code'].nunique()}")

Countries to drop: <StringArray>
['AFG', 'HTI', 'LBN', 'LBY', 'TTO']
Length: 5, dtype: str
Number of countries: 5
Remaining unique countries: 136


In [23]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita",
    "consumption_co2_per_capita", 
    "renewables_consumption"
    ]

subset_missing_2 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=2
)
display(subset_missing_2)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,renewables_consumption
country,,,,,
Bosnia and Herzegovina,0,7,0,9,9
Burundi,0,7,0,9,9
Chad,0,8,0,9,9
Gabon,0,8,0,9,9
Laos,1,8,0,0,9
Liberia,0,7,0,9,9
Mali,0,7,0,9,9
Mauritania,0,7,0,9,9
Mauritius,0,8,9,0,9


At this point we determine that any county missing all entries for the key variables should be dropped. Missing values that offer some point of reference in at least one year can be extrapolated in some way, but no reference point means that this country does not have enough info to be of real value to this dataset. Since all the above countries contain missing values for all years in the renewables_consumption variable, we drop these too.

In [28]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
]

# Find iso_codes where ANY key variable is missing for all 9 years
iso_to_drop = (
    df_raw
    .groupby("iso_code")[key_vars]
    .apply(lambda x: (x.isna().sum() == 9).any())
)

# Keep only iso_codes that do NOT satisfy the condition
df_raw = df_raw[
    ~df_raw["iso_code"].isin(iso_to_drop[iso_to_drop].index)
]

print(f"Remaining unique countries: {df_raw['iso_code'].nunique()}")

Remaining unique countries: 62


This leaves only Qatar as the final country with missing values for more than 1 key variable. So the process of removing countries without any observations for some key variable also helped in this way.

In [30]:
subset_missing_1 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=1
)
display(subset_missing_1)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Qatar,2,8,0,0,0,0,0


In [35]:
print(f"Remaining unique countries: {df_raw['iso_code'].nunique()}")

# The remaining countries are:
display(df_raw['country'].unique())
df_raw.info()

Remaining unique countries: 62


<StringArray>
['United Arab Emirates',            'Argentina',            'Australia',
              'Austria',              'Belgium',           'Bangladesh',
             'Bulgaria',              'Belarus',               'Brazil',
               'Canada',          'Switzerland',                'Chile',
                'China',             'Colombia',               'Cyprus',
              'Czechia',              'Germany',              'Denmark',
              'Ecuador',                'Egypt',                'Spain',
              'Estonia',              'Finland',               'France',
       'United Kingdom',               'Greece',              'Croatia',
              'Hungary',            'Indonesia',              'Ireland',
                 'Iran',               'Israel',                'Italy',
                'Japan',           'Kazakhstan',          'South Korea',
            'Sri Lanka',            'Lithuania',           'Luxembourg',
               'Latvia',             

<class 'pandas.DataFrame'>
Index: 558 entries, 54 to 1943
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         558 non-null    str    
 1   iso_code                        558 non-null    str    
 2   year                            558 non-null    int64  
 3   co2_per_capita                  558 non-null    float64
 4   consumption_co2_per_capita      558 non-null    float64
 5   temperature_change_from_co2     558 non-null    float64
 6   share_global_co2                558 non-null    float64
 7   land_use_change_co2_per_capita  558 non-null    float64
 8   population                      558 non-null    float64
 9   gdp                             558 non-null    float64
 10  energy_per_capita               558 non-null    float64
 11  renewables_consumption          558 non-null    float64
 12  happiness_index                 556 non-null    fl

We have substantial geopolitical and developmental heterogeneity:

- OECD core: Japan, Germany, Nordics, Netherlands, etc.
- BRICS-adjacent/emerging blocs: China, Brazil, Russia, South Africa.
- Southeast Asia: Indonesia, Thailand, Vietnam, Malaysia, Philippines.
- MENA representation: Egypt, Morocco, Qatar, UAE, Iran.
- Latin America coverage is decent.
- Eastern Europe/post-Soviet coverage is also solid.

The main structural limitation is underrepresentation of:

- Sub-Saharan Africa (outside South Africa),
- low-income economies,
- small developing states,
- and highly fragile economies.

In [ ]:
df_raw.loc[
    (df_raw["happiness_index"].isna()),
    ["country", "iso_code", "year"]
].sort_values(["country", "year"])

,country,iso_code,year
1456,Qatar,QAT,2020
1457,Qatar,QAT,2021


Let's print the differences in happiness index for QAT for available years sequentially backwards into a list to see how different the values are relative to the range of the available differences.